In [26]:
from urllib.request import urlopen
from bs4 import BeautifulSoup

def get_mission_duration(url, count = 0):
    
    # Request page
    page = urlopen(url)
    html_bytes = page.read()
    html = html_bytes.decode("utf-8")    
    soup = BeautifulSoup(html, "html.parser")
    
    # Parse HTML
    try:

        header = soup.find_all("div", {"data-testid": "timelineContainer"})[0]        
        entries = header.find_all("div", {"data-testid": "timelineDrawerTitle"})

        # print(len(entries))

        mission_length = ""
        
        for entry in entries:

            if "Missionary" in entry.text and count == 0:
            
                tag = entry.find_all("div", class_ = "sc-nsiojc-0 doIXSd")[0]        
                mission_length = "".join(tag.find_all(string=True, recursive=False)).strip()
                break

            elif "Missionary" in entry.text and count > 0:
                count -= 1

        return mission_length
      
                            
    except: # IndexError:
        print(f"Having trouble with {url}")
        return ""

url1 = "https://history.churchofjesuschrist.org/chd/individual/aage-lauritz-larsen-1894?lang=eng&timelineTabs=allTabs"
url2 = "https://history.churchofjesuschrist.org/chd/individual/aaron-benjamin-porter-jr-1875?lang=eng"
url3 = "https://history.churchofjesuschrist.org/chd/individual/aaron-burriss-dial-1899?lang=eng"



print(get_mission_duration(url1))
print(get_mission_duration(url2))
print(get_mission_duration(url3))


# PROBLEM:  what to do with missionaries who served more than one mission?


1915 January 5 – 1916 September 3 (Age 20)
1897 March 17 – 1899 July 28 (Age 21)
1920 January 27 – 1922 May 17 (Age 20)


In [27]:
import pandas as pd

df = pd.read_csv("birth_and_residence_data.csv")
df["mission"] = df["mission"].apply(lambda x: str(x).strip())

In [28]:
duration_strings = []
count = 0

for i in range(len(df)):
    cur_url = str(df["url"].iloc[i])
    if i > 0:
        prev_url = str(df["url"].iloc[i - 1])
        if prev_url == cur_url:
            count += 1
        else:
            count = 0    

    duration_strings.append(get_mission_duration(cur_url, count))

    if (i + 1) % 1000 == 0:
        print(f"Finished {i + 1} rows")

df["mission_duration"] = duration_strings
df.to_csv("duration_data.csv", index = False)

Finished 1000 rows
Finished 2000 rows
Finished 3000 rows
Finished 4000 rows
Finished 5000 rows
Finished 6000 rows
Finished 7000 rows
Finished 8000 rows
Finished 9000 rows
Finished 10000 rows
Finished 11000 rows
Having trouble with https://history.churchofjesuschrist.org/chd/individual/william-mack-harvey-1913?lang=eng&timelineTabs=allTabs
Finished 12000 rows
Finished 13000 rows
Finished 14000 rows
Finished 15000 rows
Finished 16000 rows
Finished 17000 rows
Finished 18000 rows
Finished 19000 rows
Finished 20000 rows
Finished 21000 rows
Finished 22000 rows
Finished 23000 rows
Finished 24000 rows
Finished 25000 rows
Finished 26000 rows
Finished 27000 rows
Finished 28000 rows
Finished 29000 rows
Having trouble with https://history.churchofjesuschrist.org/chd/individual/henry-dinwoody-moyle-1889?lang=eng&timelineTabs=allTabs
Having trouble with https://history.churchofjesuschrist.org/chd/individual/henry-woonsook-1895?lang=eng&timelineTabs=allTabs
Finished 30000 rows
Finished 31000 rows
Fin

### Feature Validation

In [1]:
import pandas as pd

df = pd.read_csv("duration_data.csv")


In [60]:
# sanity check
count = 0
for i in range(len(df)):
    cur_year = str(df["year"].iloc[i]).strip()
    cur_duration = df["mission_duration"].iloc[i]

    if not str(cur_year) == str(cur_duration).strip()[:4]:
        print(i, cur_year, df["url"].iloc[i])
        count += 1

print("Total:", count)
    

15454 1939 https://history.churchofjesuschrist.org/chd/individual/leo-j-muir-1880?lang=eng&timelineTabs=allTabs
15762 1937 https://history.churchofjesuschrist.org/chd/individual/mark-brimhall-garff-1907?lang=eng&timelineTabs=allTabs
29423 1909 https://history.churchofjesuschrist.org/chd/individual/henry-dinwoody-moyle-1889?lang=eng&timelineTabs=allTabs
29617 1935 https://history.churchofjesuschrist.org/chd/individual/henry-woonsook-1895?lang=eng&timelineTabs=allTabs
44479 1855 https://history.churchofjesuschrist.org/chd/individual/shepherd-pierce-hutchings-1818?lang=eng&timelineTabs=allTabs
46208 1855 https://history.churchofjesuschrist.org/chd/individual/william-wesley-willis-sr-1811?lang=eng&timelineTabs=allTabs
46360 1888 https://history.churchofjesuschrist.org/chd/individual/eli-brezee-kelsey-ferguson-1848?lang=eng&timelineTabs=allTabs
Total: 7


In [57]:
from urllib.request import urlopen
from bs4 import BeautifulSoup

def fix_mission_duration(url, cur_year):
    
    # Request page
    page = urlopen(url)
    html_bytes = page.read()
    html = html_bytes.decode("utf-8")    
    soup = BeautifulSoup(html, "html.parser")
    
    # Parse HTML
    try:

        header = soup.find_all("div", {"data-testid": "timelineContainer"})[0]        
        entries = header.find_all("div", {"data-testid": "timelineDrawerTitle"})

        # print(len(entries))

        mission_length = ""
        
        for entry in entries:

            if "Missionary" in entry.text:
            
                tag = entry.find_all("div", class_ = "sc-nsiojc-0 doIXSd")[0]  

                cur_duration = "".join(tag.find_all(string=True, recursive=False)).strip()

                if cur_duration == "":
                    # try looking for <br> tag

                    cur_duration = tag.find_all("br")[0].text


                # print(cur_duration)

                if cur_duration[:4] == str(cur_year):
                    mission_length = cur_duration
                    break


        return mission_length
      
                            
    except: # IndexError:
        print(f"Having trouble with {url}")
        return ""


# Test

url = "https://history.churchofjesuschrist.org/chd/individual/james-graham-1804?lang=eng&timelineTabs=allTabs"

print(fix_mission_duration(url, "1852"))

print(fix_mission_duration("https://history.churchofjesuschrist.org/chd/individual/john-graf-1859?lang=eng&timelineTabs=allTabs", "1895"))





ERROR! Session/line number was not unique in database. History logging moved to new session 2080
1852 August (Age 47)
1895 April 8 (Age 44)


In [59]:

durations = []

for i in range(len(df)):
    cur_year = str(df["year"].iloc[i]).strip()
    cur_duration = df["mission_duration"].iloc[i]

    if not cur_year == str(cur_duration).strip()[:4]:
        cur_duration = fix_mission_duration(df["url"].iloc[i], cur_year)
        # print(cur_year, new_duration)

    durations.append(cur_duration)

df["mission_duration"] = durations

Having trouble with https://history.churchofjesuschrist.org/chd/individual/henry-dinwoody-moyle-1889?lang=eng&timelineTabs=allTabs
Having trouble with https://history.churchofjesuschrist.org/chd/individual/henry-woonsook-1895?lang=eng&timelineTabs=allTabs
Having trouble with https://history.churchofjesuschrist.org/chd/individual/shepherd-pierce-hutchings-1818?lang=eng&timelineTabs=allTabs
Having trouble with https://history.churchofjesuschrist.org/chd/individual/eli-brezee-kelsey-ferguson-1848?lang=eng&timelineTabs=allTabs


In [62]:
df.to_csv("duration_data.csv", index = False)

### Feature Extraction

In [82]:
# Over a year vs under a year???
from random import randint

def format_time(cur_str):
    values = cur_str.split()
    try:
        if len(values) == 3:
            return pd.to_datetime(cur_str, format = "%Y %B %d")
        elif len(values) == 2:
            return pd.to_datetime(cur_str, format = "%Y %B")
        else:
            return pd.to_datetime(cur_str, format = "%Y")
    except ValueError:
        return None
        

def classify_duration(cur_year, duration_string):
    if not "–" in duration_string:
        return "Unknown"
    dates = duration_string.split("–")
    start = dates[0].strip()
    end = dates[1].strip()
    if "(" in end:
        end = end.split("(")[0].strip()

    # first check
    if not end[:4].isdecimal():
        end = f"{cur_year} {end}"

    start = format_time(start)
    end = format_time(end)

    if start == None or end == None:
        # print(duration_string)
        return 0 # end string only has a day, meaning mission lasted < 1 month?
    
    diff = (end.to_period('M') - start.to_period('M')).n

    return diff

    
index = randint(0, len(df) - 1)

cur_duration = df["mission_duration"].iloc[index]
cur_year = df["year"].iloc[index]

print(cur_duration)
print(classify_duration(cur_year, cur_duration))
    



1900 October 24 – 1903 February 10 (Age 19)
28


In [83]:
number_of_months = []

for i in range(len(df)):
    cur_duration = df["mission_duration"].iloc[i]
    cur_year = df["year"].iloc[i]

    number_of_months.append(classify_duration(cur_year, cur_duration))

df["number_of_months"] = number_of_months
df.to_csv("duration_data.csv", index = False)
    

In [4]:
count_dict = {}
number_of_months = list(df["number_of_months"])

for x in number_of_months:
    if x in count_dict:
        count_dict[x] += 1
    else:
        count_dict[x] = 1

cur_keys = list(count_dict.keys())
cur_keys.remove("Unknown")
cur_keys = sorted([int(x) for x in cur_keys])

for x in cur_keys:
    print(f"{x}: {count_dict[str(x)]}")



-1184: 1
-117: 1
-99: 1
-94: 1
-82: 1
-76: 1
-47: 1
-24: 1
-10: 1
-7: 3
-6: 1
-4: 2
-3: 1
-2: 1
-1: 2
0: 59
1: 160
2: 418
3: 660
4: 666
5: 897
6: 759
7: 511
8: 397
9: 414
10: 420
11: 412
12: 517
13: 507
14: 580
15: 511
16: 507
17: 564
18: 780
19: 860
20: 908
21: 1019
22: 1190
23: 1944
24: 4266
25: 5055
26: 4086
27: 2540
28: 1805
29: 1381
30: 1128
31: 959
32: 932
33: 716
34: 458
35: 397
36: 313
37: 240
38: 168
39: 160
40: 126
41: 107
42: 95
43: 91
44: 64
45: 55
46: 39
47: 34
48: 53
49: 36
50: 35
51: 25
52: 19
53: 8
54: 15
55: 18
56: 10
57: 3
58: 3
59: 3
60: 12
61: 2
62: 3
63: 2
64: 7
65: 8
66: 3
67: 1
68: 1
70: 2
71: 4
72: 10
73: 2
74: 1
75: 4
76: 1
78: 1
80: 2
82: 2
83: 1
84: 4
85: 1
86: 1
87: 2
89: 1
92: 1
93: 2
94: 3
96: 1
99: 1
102: 2
103: 1
105: 2
108: 2
109: 1
110: 1
116: 1
119: 1
120: 4
122: 1
123: 1
126: 1
127: 1
132: 3
137: 2
139: 2
144: 2
145: 1
148: 1
154: 1
162: 1
167: 1
168: 1
179: 1
181: 1
192: 6
197: 1
203: 1
204: 3
216: 2
219: 1
233: 1
235: 1
240: 4
252: 6
256: 1
262: 1


### Clean Up Time

Why does some durations report negative or excessively large values?

In [9]:
for i in range(len(df)):
    cur_duration = df["number_of_months"].iloc[i]
    if not cur_duration == "Unknown":
        cur_duration = int(cur_duration)
        if cur_duration == 0:
            print(cur_duration, df["mission_duration"].iloc[i])
            print(df["url"].iloc[i])

0 1939 September 14 – 13 (Age 20)
https://history.churchofjesuschrist.org/chd/individual/milton-karl-beal-1918?lang=eng&timelineTabs=allTabs
0 1939 September 14 – 13 (Age 20)
https://history.churchofjesuschrist.org/chd/individual/milton-karl-beal-1918?lang=eng&timelineTabs=allTabs
0 1900 October 8 – 28 (Age 21)
https://history.churchofjesuschrist.org/chd/individual/william-minor-leonard-1879?lang=eng&timelineTabs=allTabs
0 1855 September 21 – 22 (Age 42)
https://history.churchofjesuschrist.org/chd/individual/enoch-reese-1813?lang=eng&timelineTabs=allTabs
0 1909 June 3 – 29 (Age 19)
https://history.churchofjesuschrist.org/chd/individual/samuel-peter-jensen-1889?lang=eng&timelineTabs=allTabs
0 1926 January 5 – 12 (Age 45)
https://history.churchofjesuschrist.org/chd/individual/eugene-siverene-hansen-1880?lang=eng&timelineTabs=allTabs
0 1905 July 5 – 15 (Age 39)
https://history.churchofjesuschrist.org/chd/individual/charles-augst-ohran-ranssin-1865?lang=eng&timelineTabs=allTabs
0 1933 May 

### Test Code

In [23]:
duration_strings = []
count = 0
for i in range(100): #len(df)):
    cur_url = str(df["url"].iloc[i])
    if i > 0:
        prev_url = str(df["url"].iloc[i - 1])
        if prev_url == cur_url:
            count += 1
        else:
            count = 0    

    duration_strings.append(get_mission_duration(cur_url, count))

    if (i + 1) % 1000 == 0:
        print(f"Finished {i + 1} rows")

for d in duration_strings:
    print(d)
# df["mission_duration"] = duration_strings
# df.to_csv("duration_data.csv", index = False)

ERROR! Session/line number was not unique in database. History logging moved to new session 2079
1931 September 16 – 1932 September 22 (Age 19)
1898 March 30 – October 1 (Age 51)
1919 July 8 – 1921 August 27 (Age 23)
1912 July 23 – 1914 September 3 (Age 22)
1935 January 16 – 1937 January 15 (Age 40)
1919 May 27 – 1921 July 23 (Age 21)
1913 September 24 – 1915 July 7 (Age 20)
1900 September 5 – 1902 August 23 (Age 28)
1924 October 7 – 1926 November 1 (Age 26)
1927 December 6 – 1930 January 30 (Age 21)
1925 February 26 – 1927 February 26 (Age 21)
1929 June 4 – 1931 July 31 (Age 22)
1909 November 23 – 1912 April 1 (Age 24)
1921 May 25 – 1924 May 7 (Age 24)
1899 June 14 – 1901 May 15 (Age 35)
1925 November 24 – 1926 April 1 (Age 62)
1937 February 17 – May 7 (Age 57)
1897 August 6 (Age 22)
1940 January 31 – 1942 February 3 (Age 19)
1899 April 19 – 1901 June 19 (Age 20)
1913 September 10 – 1914 April 21 (Age 33)
1914 October 27 – 1915 May 2 (Age 35)
1880 April 12 – 1882 July 10 (Age 33)
1929

In [24]:
df["url"].iloc[97]

'https://history.churchofjesuschrist.org/chd/individual/alma-golden-andrus-1899?lang=eng&timelineTabs=allTabs'